  EDA — Eksplorasi Data: ontime_features
  Tujuan: Membuktikan secara ilmiah bahwa dataset imbalance
          dan memahami distribusi data sebelum modelling
===========================================================

### 1. IMPORT LIBRARY

In [1]:
import clickhouse_connect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

In [2]:
# ── Gaya visual konsisten ──────────────────────────────────
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})

In [3]:
# ──────────────────────────────────────────────────────────
# 1. KONEKSI KE CLICKHOUSE
# ──────────────────────────────────────────────────────────
 
client = clickhouse_connect.get_client(
    host="13.215.79.3",
    port=8123,
    username="default",
    password="rahasia123",
    database="flight_delay"
)

In [4]:
# ──────────────────────────────────────────────────────────
# 2. AMBIL SAMPEL DATA (efisien, tidak menarik semua baris)
# ──────────────────────────────────────────────────────────
print("\n[1/7] Mengambil sampel data dari server...")

# Ambil 1 juta baris untuk EDA; cepat dan representatif
df = client.query_df("""
    SELECT
        FlightDate,
        flight_year,
        flight_month,
        day_of_week,
        dep_hour,
        dep_time_bucket,
        season,
        IATA_CODE_Reporting_Airline,
        Origin,
        Dest,
        Distance,
        route_avg_arr_delay_prev,
        carrier_arr_delay_rate_prev,
        origin_arr_delay_rate_prev,
        dest_arr_delay_rate_prev,
        arr_del15_label,
        dep_del15_label,
        cancelled_label
    FROM ontime_features
""")

print(f"   ✅ Data berhasil diambil: {len(df):,} baris, {df.shape[1]} kolom")




[1/7] Mengambil sampel data dari server...
   ✅ Data berhasil diambil: 29,208,900 baris, 18 kolom


In [5]:
# ──────────────────────────────────────────────────────────
# 3. INFO UMUM DATASET
# ──────────────────────────────────────────────────────────
print("\n[2/7] Ringkasan dataset...")
print(f"\n{'─'*40}")
print(f"  Jumlah baris   : {len(df):,}")
print(f"  Jumlah kolom   : {df.shape[1]}")
print(f"  Rentang tahun  : {df['flight_year'].min()} – {df['flight_year'].max()}")
print(f"  Maskapai unik  : {df['IATA_CODE_Reporting_Airline'].nunique()}")
print(f"  Bandara asal   : {df['Origin'].nunique()}")
print(f"  Bandara tujuan : {df['Dest'].nunique()}")
print(f"{'─'*40}")

# Cek missing values
print("\n📌 Nilai NULL per kolom (hanya kolom yang ada NULL-nya):")
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if null_counts.empty:
    print("   Tidak ada nilai NULL.")
else:
    for col, cnt in null_counts.items():
        print(f"   {col:40s}: {cnt:,} ({cnt/len(df):.1%})")




[2/7] Ringkasan dataset...

────────────────────────────────────────
  Jumlah baris   : 29,208,900
  Jumlah kolom   : 18
  Rentang tahun  : 2021 – 2025
  Maskapai unik  : 17
  Bandara asal   : 380
  Bandara tujuan : 381
────────────────────────────────────────

📌 Nilai NULL per kolom (hanya kolom yang ada NULL-nya):
   Tidak ada nilai NULL.


In [6]:
# ──────────────────────────────────────────────────────────
# 4. ANALISIS IMBALANCE — BUKTI UTAMA
# ──────────────────────────────────────────────────────────
print("\n[3/7] Analisis distribusi kelas (imbalance check)...")

# Hitung distribusi
label = df["arr_del15_label"].fillna(0).astype(int)
counts = label.value_counts().sort_index()
total = counts.sum()
pct = counts / total * 100

print(f"\n{'─'*50}")
print(f"  {'Kelas':<20} {'Jumlah':>12} {'Proporsi':>10}")
print(f"{'─'*50}")
print(f"  {'0 = Tepat Waktu (ontime)':<20} {counts[0]:>12,} {pct[0]:>9.2f}%")
print(f"  {'1 = Delay (≥15 menit)':<20} {counts[1]:>12,} {pct[1]:>9.2f}%")
print(f"{'─'*50}")
imbalance_ratio = counts[0] / counts[1]
print(f"  Rasio imbalance: {imbalance_ratio:.2f}:1 (ontime vs delay)")
print(f"{'─'*50}")

if imbalance_ratio >= 2:
    print(f"\n  ⚠️  KONFIRMASI: Dataset IMBALANCE (rasio {imbalance_ratio:.1f}:1)")
    print("     Model yang dilatih tanpa penanganan khusus akan bias")
    print("     memprediksi kelas mayoritas (Tepat Waktu).")
else:
    print("\n  ✅ Dataset relatif seimbang.")




[3/7] Analisis distribusi kelas (imbalance check)...

──────────────────────────────────────────────────
  Kelas                      Jumlah   Proporsi
──────────────────────────────────────────────────
  0 = Tepat Waktu (ontime)   23,364,759     79.99%
  1 = Delay (≥15 menit)    5,844,141     20.01%
──────────────────────────────────────────────────
  Rasio imbalance: 4.00:1 (ontime vs delay)
──────────────────────────────────────────────────

  ⚠️  KONFIRMASI: Dataset IMBALANCE (rasio 4.0:1)
     Model yang dilatih tanpa penanganan khusus akan bias
     memprediksi kelas mayoritas (Tepat Waktu).


In [7]:
# ──────────────────────────────────────────────────────────
# 5. VISUALISASI — SEMUA PLOT DALAM SATU FILE
# ──────────────────────────────────────────────────────────
print("\n[4/7] Membuat visualisasi...")

fig = plt.figure(figsize=(18, 20))
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Distribusi kelas (Bar) ──────────────────────
ax1 = fig.add_subplot(gs[0, 0])
colors = ["#4CAF50", "#F44336"]
bars = ax1.bar(["Tepat Waktu (0)", "Delay (1)"],
               [counts[0], counts[1]], color=colors, edgecolor="white", linewidth=1.2)
for bar, pct_val in zip(bars, pct):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + total * 0.005,
             f"{pct_val:.1f}%\n({counts[int(bar.get_x() > 0.5)]:,})",
             ha="center", va="bottom", fontweight="bold", fontsize=11)
ax1.set_title("Distribusi Kelas Label (arr_del15_label)")
ax1.set_ylabel("Jumlah Penerbangan")
ax1.set_ylim(0, counts.max() * 1.18)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── Plot 2: Pie chart kelas ──────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
wedge_props = {"linewidth": 2, "edgecolor": "white"}
ax2.pie([counts[0], counts[1]],
        labels=["Tepat Waktu", "Delay"],
        autopct="%1.1f%%",
        colors=colors,
        wedgeprops=wedge_props,
        startangle=90,
        textprops={"fontsize": 12})
ax2.set_title(f"Proporsi Kelas\n(Rasio Imbalance ≈ {imbalance_ratio:.1f}:1)")

# ── Plot 3: Distribusi delay per tahun ──────────────────
ax3 = fig.add_subplot(gs[1, :])
yearly = (df.groupby("flight_year")["arr_del15_label"]
            .value_counts(normalize=True)
            .rename("proportion")
            .reset_index())
yearly.columns = ["flight_year", "arr_del15_label", "proportion"]
delay_yearly = yearly[yearly["arr_del15_label"] == 1]
ontime_yearly = yearly[yearly["arr_del15_label"] == 0]

x = np.arange(len(delay_yearly["flight_year"]))
width = 0.35
ax3.bar(x - width/2, ontime_yearly["proportion"].values * 100,
        width, label="Tepat Waktu", color="#4CAF50", alpha=0.85)
ax3.bar(x + width/2, delay_yearly["proportion"].values * 100,
        width, label="Delay", color="#F44336", alpha=0.85)
ax3.set_xticks(x)
ax3.set_xticklabels(delay_yearly["flight_year"].astype(int))
ax3.set_title("Proporsi Tepat Waktu vs Delay per Tahun")
ax3.set_ylabel("Proporsi (%)")
ax3.set_xlabel("Tahun")
ax3.legend()
ax3.set_ylim(0, 105)
for i, (ot, dl) in enumerate(zip(ontime_yearly["proportion"].values,
                                  delay_yearly["proportion"].values)):
    ax3.text(i - width/2, ot * 100 + 1, f"{ot*100:.1f}%",
             ha="center", fontsize=9)
    ax3.text(i + width/2, dl * 100 + 1, f"{dl*100:.1f}%",
             ha="center", fontsize=9)

# ── Plot 4: Delay per jam keberangkatan ──────────────────
ax4 = fig.add_subplot(gs[2, 0])
hourly_delay = (df.groupby("dep_hour")["arr_del15_label"]
                  .mean()
                  .reset_index())
hourly_delay.columns = ["dep_hour", "delay_rate"]
ax4.plot(hourly_delay["dep_hour"], hourly_delay["delay_rate"] * 100,
         marker="o", color="#E91E63", linewidth=2, markersize=5)
ax4.fill_between(hourly_delay["dep_hour"],
                 hourly_delay["delay_rate"] * 100, alpha=0.15, color="#E91E63")
ax4.set_title("Tingkat Delay per Jam Keberangkatan")
ax4.set_xlabel("Jam Keberangkatan (0–23)")
ax4.set_ylabel("Delay Rate (%)")
ax4.set_xticks(range(0, 24, 2))

# ── Plot 5: Delay per hari dalam seminggu ────────────────
ax5 = fig.add_subplot(gs[2, 1])
day_labels = ["Sen", "Sel", "Rab", "Kam", "Jum", "Sab", "Min"]
weekly_delay = (df.groupby("day_of_week")["arr_del15_label"]
                  .mean()
                  .reindex(range(1, 8))
                  .reset_index())
weekly_delay.columns = ["day_of_week", "delay_rate"]
bar_colors = ["#F44336" if v == weekly_delay["delay_rate"].max()
              else "#2196F3" for v in weekly_delay["delay_rate"]]
ax5.bar(day_labels, weekly_delay["delay_rate"] * 100,
        color=bar_colors, edgecolor="white")
ax5.set_title("Tingkat Delay per Hari dalam Seminggu\n(merah = hari paling delay)")
ax5.set_ylabel("Delay Rate (%)")
ax5.set_xlabel("Hari")

# ── Plot 6: Delay per musim ───────────────────────────────
ax6 = fig.add_subplot(gs[3, 0])
season_order = ["Winter", "Spring", "Summer", "Fall"]
season_delay = (df.groupby("season")["arr_del15_label"]
                  .mean()
                  .reindex(season_order)
                  .fillna(0)
                  .reset_index())
season_delay.columns = ["season", "delay_rate"]
ax6.bar(season_delay["season"], season_delay["delay_rate"] * 100,
        color=["#1976D2", "#43A047", "#FB8C00", "#795548"], edgecolor="white")
ax6.set_title("Tingkat Delay per Musim")
ax6.set_ylabel("Delay Rate (%)")
ax6.set_xlabel("Musim")
for i, v in enumerate(season_delay["delay_rate"]):
    ax6.text(i, v * 100 + 0.3, f"{v*100:.1f}%", ha="center", fontsize=10)

# ── Plot 7: Top 10 maskapai berdasarkan delay rate ────────
ax7 = fig.add_subplot(gs[3, 1])
carrier_stats = (df.groupby("IATA_CODE_Reporting_Airline")["arr_del15_label"]
                   .agg(["mean", "count"])
                   .reset_index())
carrier_stats.columns = ["carrier", "delay_rate", "count"]
carrier_stats = carrier_stats[carrier_stats["count"] >= 1000]
top10 = carrier_stats.nlargest(10, "delay_rate")
ax7.barh(top10["carrier"], top10["delay_rate"] * 100,
         color="#FF7043", edgecolor="white")
ax7.set_title("Top 10 Maskapai: Delay Rate Tertinggi")
ax7.set_xlabel("Delay Rate (%)")
ax7.set_ylabel("Kode Maskapai")
ax7.invert_yaxis()
for i, v in enumerate(top10["delay_rate"]):
    ax7.text(v * 100 + 0.2, i, f"{v*100:.1f}%", va="center", fontsize=9)

# ── Judul utama ────────────────────────────────────────────
fig.suptitle(
    "EDA — Dataset ontime_features: Analisis Distribusi & Imbalance",
    fontsize=15, fontweight="bold", y=1.01
)

plt.savefig("eda_ontime_features2.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ Plot disimpan ke eda_ontime_features.png")




[4/7] Membuat visualisasi...
   ✅ Plot disimpan ke eda_ontime_features.png


In [8]:
# ──────────────────────────────────────────────────────────
# 6. STATISTIK DESKRIPTIF FITUR NUMERIK
# ──────────────────────────────────────────────────────────
print("\n[5/7] Statistik deskriptif fitur numerik...")
num_cols = ["Distance", "route_avg_arr_delay_prev",
            "carrier_arr_delay_rate_prev",
            "origin_arr_delay_rate_prev",
            "dest_arr_delay_rate_prev"]
print(df[num_cols].describe().round(3).to_string())




[5/7] Statistik deskriptif fitur numerik...
           Distance  route_avg_arr_delay_prev  carrier_arr_delay_rate_prev  origin_arr_delay_rate_prev  dest_arr_delay_rate_prev
count  2.920890e+07              2.920890e+07                 2.920890e+07                2.920890e+07              2.920890e+07
mean   8.273620e+02              1.516900e+01                 1.990000e-01                1.990000e-01              1.990000e-01
std    5.959320e+02              8.588000e+00                 5.900000e-02                5.800000e-02              5.400000e-02
min    1.100000e+01              0.000000e+00                 0.000000e+00                0.000000e+00              0.000000e+00
25%    3.950000e+02              9.564000e+00                 1.640000e-01                1.630000e-01              1.670000e-01
50%    6.780000e+02              1.421600e+01                 1.970000e-01                2.020000e-01              2.030000e-01
75%    1.065000e+03              1.931200e+01       

In [9]:
# ──────────────────────────────────────────────────────────
# 7. PERBANDINGAN FITUR HISTORIS: ONTIME vs DELAY
# ──────────────────────────────────────────────────────────
print("\n[6/7] Perbandingan rata-rata fitur historis: Ontime vs Delay...")

compare_cols = ["route_avg_arr_delay_prev",
                "carrier_arr_delay_rate_prev",
                "origin_arr_delay_rate_prev",
                "dest_arr_delay_rate_prev",
                "Distance"]

comparison = (df.groupby("arr_del15_label")[compare_cols]
                .mean()
                .T
                .rename(columns={0: "Tepat Waktu (0)", 1: "Delay (1)"}))
comparison["Selisih"] = comparison["Delay (1)"] - comparison["Tepat Waktu (0)"]
print("\n", comparison.round(4).to_string())

print("\n💡 Interpretasi:")
print("   Penerbangan yang delay cenderung memiliki:")
print("   - route_avg_arr_delay_prev lebih tinggi (rute bermasalah)")
print("   - carrier_arr_delay_rate_prev lebih tinggi (maskapai kurang tepat waktu)")
print("   - origin/dest_arr_delay_rate_prev lebih tinggi (bandara sibuk)")




[6/7] Perbandingan rata-rata fitur historis: Ontime vs Delay...

 arr_del15_label              Tepat Waktu (0)  Delay (1)  Selisih
route_avg_arr_delay_prev             14.4729    17.9534   3.4805
carrier_arr_delay_rate_prev           0.1949     0.2167   0.0218
origin_arr_delay_rate_prev            0.1952     0.2163   0.0211
dest_arr_delay_rate_prev              0.1956     0.2139   0.0183
Distance                            820.4325   855.0644  34.6319

💡 Interpretasi:
   Penerbangan yang delay cenderung memiliki:
   - route_avg_arr_delay_prev lebih tinggi (rute bermasalah)
   - carrier_arr_delay_rate_prev lebih tinggi (maskapai kurang tepat waktu)
   - origin/dest_arr_delay_rate_prev lebih tinggi (bandara sibuk)


In [10]:
# ──────────────────────────────────────────────────────────
# 8. RINGKASAN TEMUAN
# ──────────────────────────────────────────────────────────
print("\n[7/7] RINGKASAN TEMUAN EDA")
print("=" * 60)
print(f"""
📊 DATASET:
   • Sampel yang dianalisis : {len(df):,} baris
   • Rentang tahun          : {df['flight_year'].min()} – {df['flight_year'].max()}
   • Maskapai               : {df['IATA_CODE_Reporting_Airline'].nunique()} maskapai
   • Bandara asal           : {df['Origin'].nunique()} bandara

⚖️  IMBALANCE:
   • Tepat Waktu (0): {pct[0]:.1f}%  ({counts[0]:,} penerbangan)
   • Delay (1)      : {pct[1]:.1f}%  ({counts[1]:,} penerbangan)
   • Rasio          : {imbalance_ratio:.2f}:1 → Dataset IMBALANCE ✅ terbukti

🔍 TEMUAN LAIN:
   • Delay rate tertinggi ada di jam sore-malam (efek cascading)
   • Delay lebih tinggi di Winter & Summer
   • Fitur historis (route_avg_arr_delay_prev, dst.) berbeda
     signifikan antara penerbangan ontime vs delay → fitur kuat!
""")
print("=" * 60)
print("EDA selesai. Cek file: eda_ontime_features.png")


[7/7] RINGKASAN TEMUAN EDA

📊 DATASET:
   • Sampel yang dianalisis : 29,208,900 baris
   • Rentang tahun          : 2021 – 2025
   • Maskapai               : 17 maskapai
   • Bandara asal           : 380 bandara

⚖️  IMBALANCE:
   • Tepat Waktu (0): 80.0%  (23,364,759 penerbangan)
   • Delay (1)      : 20.0%  (5,844,141 penerbangan)
   • Rasio          : 4.00:1 → Dataset IMBALANCE ✅ terbukti

🔍 TEMUAN LAIN:
   • Delay rate tertinggi ada di jam sore-malam (efek cascading)
   • Delay lebih tinggi di Winter & Summer
   • Fitur historis (route_avg_arr_delay_prev, dst.) berbeda
     signifikan antara penerbangan ontime vs delay → fitur kuat!

EDA selesai. Cek file: eda_ontime_features.png
